# Persistence - Disk

## 🟢 Disk Driver Overview
### Step 1: Wait for drive to be ready
The driver reads the status register until:
- drive is READY
- drive is not BUSY

This prevents sending a new command at the wrong time.

So before issuing commands, the driver synchronizes with device state.

### Step 2: Write command parameters
The driver writes:
- sector count
- logical block address (LBA)
- drive selection
- other needed command fields

These go into command-related registers.

This is the “describe what I want” phase.

### Step 3: Start the I/O
The driver writes the actual READ or WRITE command into the command register.

At this point the controller begins the operation.

This is the “go do it” phase.

### Step 4: Data transfer
For writes:
- the device may signal it is ready to accept data
- the driver writes data to the data port

For reads:
- data may later be read from the device port, or moved via DMA depending on design

This is the “move the bytes” phase.

### Step 5: Handle interrupts
The device signals completion, partial completion, or sector progress depending on hardware behavior.

The driver then:
- acknowledges the interrupt
- checks status
- continues transfer if needed
- completes the request
- wakes waiters

This is the “finish and notify the OS” phase.

### Step 6: Error handling
After operations, the driver checks status and possibly a dedicated error register.

If something went wrong:
- retry
- fail request
- log error
- reset device if necessary

This is important because real hardware is unreliable enough that drivers must expect failures.

---

## 🟢 Disks: what the OS is optimizing around

A traditional disk is organized into:
- platters
- surfaces
- tracks on each surface
- sectors on each track

A cylinder is the set of tracks at the same radius across all platter surfaces. If the arm moves to one cylinder, all heads are aligned to that cylinder together. Disk I/O transfers data in sector-sized chunks; a read/write is effectively atomic at the sector level. The total access time has three main parts:
- **seek time:** move the head to the right track/cylinder
- **rotational latency:** wait for the desired sector to rotate under the head
- **transfer time:** actually read or write the bytes

That is why disks are much slower than CPUs, and why the OS uses queues, caching, and typically DMA rather than per-word programmed I/O.

### Why sector numbering and layout matter
In memory “nearby logical blocks” should ideally also be “nearby physically,” so sequential access can stream with fewer delays. They also show interleaving: older systems sometimes left gaps between logically consecutive sectors so the CPU/controller had enough time to process one sector before the next relevant one arrived.

The disk-scheduling are not just about algorithms; The tradeoffs are **fairness**, **throughput**, and **response time**. The stated goals are **no starvation**, **fairness**, **high throughput**, **low mean response time**, and **predictability**. Those goals conflict, which is why no single policy is “best” in every workload.

---

## 🟢 Disk scheduling
If many requests are queued, the OS can reorder them to reduce head movement.
**Goals**
- No starvation
- Fairness
- High throughput
- Low mean response time
- Predictability


### FCFS
**First-Come, First-Served** is fair and simple, but it can be terrible for performance because the head may bounce randomly across the disk.

It's the simplest and fairest because requests are handled in arrival order. Its weakness is that it preserves a random seek pattern: if requests arrive for cylinders far apart, the head may repeatedly cross the disk. In the example sequence `10, 3, 11, 6` starting from cylinder `5`, FCFS incurs total movement `5+7+8+5 = 25`. The point is not the arithmetic; it is that fairness does not imply efficiency.

### SSTF
Shortest-Seek-Time-First picks the next request requiring the smallest head movement. On the same example it reduces movement to 12, so average performance improves. But SSTF can starve distant requests if new nearby requests keep arriving. It is locally optimal, not globally fair.

### SCAN/Elevator
The head moves in one direction, servicing everything on the way, then reverses. This improves fairness compared with SSTF and still reduces seeks.

### C-SCAN
The head services requests in one direction only, then jumps back to the start. This gives more uniform wait times.

### N-step SCAN
Requests are batched; the current sweep services one batch while new arrivals wait for the next batch. This avoids the queue changing chaotically during a sweep.

#### The important theme is the tradeoff:
- FCFS: fairer, slower
- SSTF: fast average seek, possible starvation
- SCAN/C-SCAN: better balance of throughput and fairness

**SCAN** and **C-SCAN** are attempts to keep SSTF’s efficiency while restoring more predictable wait times. SCAN behaves like an elevator: service requests while sweeping in one direction, then reverse. C-SCAN services in one direction only, then jumps back, making waiting times more uniform because requests are not advantaged merely by being “behind” the head. N-step SCAN adds batching: new arrivals are deferred to the next sweep, which prevents the active queue from constantly changing mid-pass.

---

## 🟢 System Level Disk Consideration
Scheduling policy cannot be judged in isolation. As multiprogramming increases, disk load increases too. Access patterns are also highly non-uniform: `ls` and `du` stress storage differently. Performance is affected not only by scheduling, but also by multiple disk subsystems/RAID, logical file organization, caching, fragmentation, and compression. That slide is really saying: the disk scheduler is only one layer in a larger storage-performance story.

---

## 🟢 File systems: what problem they solve
The file-system slides deliberately contrast two views of a file. From the user view, a file is just related information created for some purpose: program, text, spreadsheet, database. From the OS view, a file is a logical storage unit mapped onto physical storage. That difference matters because the OS does not reason in terms of “documents”; it reasons in terms of blocks, metadata, names, and allocation state.

File handling include:
- **create:** allocate metadata and often space, then add a directory entry
- **read/write:** resolve the name, open the file, translate logical offsets into physical blocks, and maintain current file position
- **remove:** delete the directory entry and deallocate blocks

A file system gives users a simple abstraction:
- named files
- directories
- operations like open, read, write, close, remove

But underneath, the OS must answer:
- Where are this file’s blocks on disk?
- How are directories represented?
- Where is metadata stored?
- Which blocks are free?

So a file system is really a mapping from: human-friendly names and byte streams to specific disk blocks and metadata structures.

---

## 🟢 Basic file-system layout on disk
- superblock: global info about the file system
- inode area: metadata per file
- allocation structures: bitmaps/free lists
- data blocks: actual file contents
- sometimes a boot block

### Superblock
Contains high-level facts:
- size of the file system
- number/location of inodes
- number/location of data blocks
- “magic number” identifying the format

### Inodes
An inode stores metadata for one file:
- owner
- permissions
- size
- timestamps
- link count
- pointers to data blocks

The key idea is:
- directory entries map names to inode numbers, and the inode then tells where the file’s content lives.

---

## 🟢 Inodes and Block Addressing
The ideal would be storing each file sequentially, but that is impractical. The solution is that the inode stores pointers to the blocks used by the file. This is the key Unix idea: logical continuity does not require physical contiguity.

The addressing scheme is layered:
- 10 direct pointers: fast, compact, ideal for small files
- single indirect: one pointer block adds 256 more data-block pointers
- double indirect: one more level expands to 256×256
- triple indirect: one more level expands to 256×256×256

With 1 KB blocks and 4-byte pointers, one indirect block holds 256 pointers. That yields the capacity progression shown on the slide:
- direct: about 10 KB
- single indirect: about 266 KB
- double indirect: about 64 MB
- triple indirect: about 16 GB

The deeper point is why this design works well: most files are small, so they use only direct pointers and are cheap to access; a minority are large, and the indirect levels allow them without wasting inode space on every file.

---

## 🟢 File Access Methods - Sequential and Direct
**Sequential access** reads or writes consecutive blocks efficiently. **Direct access** allows reads/writes in arbitrary order, like `seek()` in Unix. This matters because a data organization that is excellent for one may be poor for the other.

The 128 MB database given as example. If records are variable-length and separated by delimiters, then:
- finding record 555555555 is hard unless you have auxiliary indexing
- inserting a record may require moving large regions if you insist on keeping sorted physical order

Logical organization of a file can dominate performance as much as raw disk speed. A file is not automatically a database just because it stores records.

---

## 🟢 Directories

Directories are implemented as special files containing entries like:
- filename
- inode number

So pathname lookup such as `/etc/passwd` is really:
- start at root directory
- find `etc`
- follow its inode
- scan etc directory entries
- find `passwd`

follow its inode to the actual data blocks

That is why opening a file may require several separate disk reads if metadata and directories are scattered. The slide showing multiple block numbers for `/etc/passwd` is making exactly this point: metadata locality matters a lot.

---

## 🟢 How file data is addressed

The inode cannot usually store every block pointer for huge files directly, so Unix-style systems use a layered scheme:

### Direct pointers
The inode stores pointers directly to the first few data blocks. This is fast and efficient for small files.

### Single indirect
The inode points to a block full of more pointers.

### Double indirect
The inode points to a block of pointers to pointer blocks.

### Triple indirect
Adds one more level for very large files.

This design is clever because:
- most files are small, so direct pointers are enough and access is fast
- large files still work, using indirect blocks when needed

The cost is that very large files need more pointer traversals.

---

## 🟢 Alternative block-addressing strategies
### Inode pointer scheme
Good random access, good for mixed small/large files.

### Extent-based
Store a file as “start block + length.” Very efficient for contiguous files, but hard if free space is fragmented.

### Linked allocation / FAT-like
Each block points to the next. Easy to grow, but random access is slower because you may have to follow many links.

So there is no perfect scheme:
- extents are great for large contiguous runs
- linked schemes are simple but weak for random access
- inode-pointer schemes are a practical middle ground

---

## 🟢 Fragmentation
### Internal fragmentation
Space wasted inside an allocated block.

Example: file uses only part of its last block.

Smaller blocks reduce this, but then the file system must manage and transfer more blocks.

### External fragmentation
Free space exists, but in scattered holes.

This hurts contiguous allocation and may require defragmentation.

This is why file-system design is always a balance between:
- large blocks for speed
- small blocks for space efficiency

---

## 🟢 Why FFS was a big improvement

The Fast File System (FFS) idea is:
- be disk-aware when placing metadata and data.

Instead of scattering everything arbitrarily, FFS groups nearby disk space into cylinder groups (more generally, locality groups of blocks).

Within a group, it tries to place near each other:
- a file’s inode and its data blocks
- files in the same directory
- related directory data

That reduces long seeks during common operations like:
- pathname lookup
- reading a file after opening it
- traversing a directory tree

The contrast slide between “non-FFS” and “FFS” is showing exactly that:
- non-FFS may spread related objects across many groups
- FFS deliberately clusters related objects to improve locality

### Large-file exception

A very large file can fill a whole group and crowd out related files. FFS handles this by spreading later chunks of a large file into other groups, but in larger chunks so the seek cost is amortized.

---

## 🟢 Log-structured file systems (LFS)

LFS starts from a different observation:
- modern workloads often do many writes
- sequential disk writes are much faster than random writes

So instead of updating metadata and data in place all over the disk, LFS:
- buffers many updates in memory
- writes them out together as a large sequential log segment

This can make write performance much better, because one logical file update often touches many structures:
- data block
- inode
- directory block
- allocation bitmap
- etc.

Writing those one by one in place causes many small random writes. LFS turns them into fewer large sequential writes. The price is more complexity later in cleaning/reclaiming old segments.